### PNLP Dataset update

**Auteur** : Esteban Montandon

**Date** : Mars 2024

**Pays** : RDC

In [ ]:
# !pip install --upgrade pyarrow # Environment change errors
%conda install -c conda-forge psycopg2-binary -y -q

In [ ]:
%conda install -c conda-forge pyarrow -y -q

In [ ]:
import os 
import pandas as pd
from datetime import datetime
import re
from openhexa.sdk.workspaces import workspace
from io import StringIO
import tempfile
from sqlalchemy import create_engine, text

import pyarrow ##

### Nom du dernier fichier téléchargé

In [ ]:
# parameters
# nouvelles_donnees_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2024/2024_Database_Week24.csv"
# nouvelles_donnees_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2026/2026_Database_Week05.csv"

In [ ]:
# Set DB engine
engine = create_engine(workspace.database_url)

In [ ]:
# extrait l'année et la semaine du nom du fichier
nouvelles_fishier_nom = os.path.basename(nouvelles_donnees_path) 
nouvelle_version = nouvelles_fishier_nom.split('.')[0]
print(f"Dernier fichier téléchargé: {nouvelles_fishier_nom}")

In [ ]:
# Nom de la nouvelle version 
date_version = f"{nouvelle_version.lower().replace('database_','')}_{datetime.now().strftime('%Y_%m_%d_%H%M')}"
print(f"Nouvelle version: {date_version}")

### Chargement du tableau: DSE_palu_for_PNLP

In [ ]:
# Historical date
DSE_palu_for_PNLP_table = pd.read_sql_table('DSE_palu_for_PNLP',con=engine)
max_date = datetime.strptime(str(DSE_palu_for_PNLP_table.Date.max()), '%Y-%m-%d %H:%M:%S') 
dse_palu_max_week = DSE_palu_for_PNLP_table[DSE_palu_for_PNLP_table.Date >= datetime(max_date.year, 1, 1, 0, 0)].NUMSEM.max()
print(f"DSE_palu_for_PNLP dernière semaine disponible: {max_date.year} - {dse_palu_max_week}")
# DSE_palu_for_PNLP_table.head(5)

### Chargement du tableau: DSE_palu_for_PNLP_2022

In [ ]:

DSE_palu_for_PNLP_2022_table = pd.read_sql_table('DSE_palu_for_PNLP_2022',con=engine)
max_date = datetime.strptime(str(DSE_palu_for_PNLP_2022_table.Date.max()), '%Y-%m-%d %H:%M:%S') 
dse_palu_max_week = DSE_palu_for_PNLP_2022_table[DSE_palu_for_PNLP_2022_table.Date >= datetime(max_date.year, 1, 1, 0, 0)].NUMSEM.max()
print(f"DSE_palu_for_PNLP_2022 dernière semaine disponible: {max_date.year} - {dse_palu_max_week}")
# DSE_palu_for_PNLP_2022_table.head(5)

### Chargement du tableau: DSE_palu_for_PNLP_tot

In [ ]:
# Chargement du tableau
DSE_palu_for_PNLP_tot_table = pd.read_sql_table('DSE_palu_for_PNLP_tot',con=engine)
max_date = datetime.strptime(str(DSE_palu_for_PNLP_tot_table.Date.max()), '%Y-%m-%d %H:%M:%S') 
dse_palu_max_week = DSE_palu_for_PNLP_tot_table[DSE_palu_for_PNLP_tot_table.Date >= datetime(max_date.year, 1, 1, 0, 0)].NUMSEM.max()
print(f"DSE_palu_for_PNLP_tot dernière semaine disponible: {max_date.year} - {dse_palu_max_week}")
# DSE_palu_for_PNLP_tot_table.head(5)

### Chargement du tableau: DSE_Completude

In [ ]:
# Chargement du tableau DSE_Completude
DSE_Completude_table = pd.read_sql_table('DSE_Completude',con=engine)
dse_comp_week_available = DSE_Completude_table[DSE_Completude_table['year'] == DSE_Completude_table.year.max()].NUMSEM.max()
print(f"DSE_Completude dernière semaine disponible: {DSE_Completude_table.year.unique().max()} - {dse_comp_week_available}")
# DSE_Completude_table.head(5)

### Créer une nouvelle version de dataset

In [ ]:
date_version

In [ ]:
# Obtenir le dataset 
dataset = workspace.get_dataset("pnlp-weekly-malaria-dat-4ae811")

try:
    # Créer une nouvelle version
    version = dataset.create_version(date_version) # this might throw an exception if the name exists
except Exception as e:
    print("La version du dataset deja existe")
    print(f"ERROR: {e}")
    

In [ ]:
# Ajouter les fichiers au dataset

try:

    # Ajouter DSE_palu_for_PNLP .parquet
    with tempfile.NamedTemporaryFile(suffix=".parquet") as tmp:
        DSE_palu_for_PNLP_table.to_parquet(tmp.name)
        version.add_file(tmp.name, filename=f"DSE_palu_for_PNLP_{date_version}.parquet")

    # Ajouter DSE_palu_for_PNLP_2022 .parquet
    with tempfile.NamedTemporaryFile(suffix=".parquet") as tmp:
        DSE_palu_for_PNLP_2022_table.to_parquet(tmp.name)
        version.add_file(tmp.name, filename=f"DSE_palu_for_PNLP_2022_{date_version}.parquet")

    # Ajouter DSE_palu_for_PNLP_tot .parquet
    with tempfile.NamedTemporaryFile(suffix=".parquet") as tmp:
        DSE_palu_for_PNLP_tot_table.to_parquet(tmp.name)
        version.add_file(tmp.name, filename=f"DSE_palu_for_PNLP_tot_{date_version}.parquet")
        
    # Ajouter DSE_Completude .parquet
    with tempfile.NamedTemporaryFile(suffix=".parquet") as tmp:
        DSE_Completude_table.to_parquet(tmp.name)
        version.add_file(tmp.name, filename=f"DSE_Completude_{date_version}.parquet")

    print(f"Nouvelle version du dataset {date_version} créée")
    
except Exception as e:
    print(e)
    raise


In [ ]:
# helper openhexa code in: 
# https://github.com/BLSQ/openhexa/wiki/Using-the-OpenHEXA-SDK#working-with-datasets


In [ ]:
# include index for table "DSE_palu_for_PNLP_tot"
create_index_query = """CREATE INDEX IF NOT EXISTS idx_dse_palu_year_week_zone ON public."DSE_palu_for_PNLP_tot" ("year", "NUMSEM", "Nom");"""
with engine.begin() as conn:  # auto-commits the transaction
    conn.execute(text(create_index_query))
